# Laboratorium SI: wnioskowanie regułowe w ocenie ryzyka transakcji kartowych

Projekt: **Wykrywanie anomalii w transakcjach kartowych (fraud detection)**. Ten notebook (zadania G1–G3, B1, P1) buduje **system regułowy**, który z cech transakcji wyprowadza poziom ryzyka oraz zalecenie dla analityka antyfraudowego: zatwierdź, eskaluj (kontakt z klientem) albo zablokuj do weryfikacji.

Cykl wnioskowania w przód:
1. **Fakty** – cechy transakcji (np. wysoka kwota, nowe urządzenie)
2. **Dopasowanie reguł (agenda)** – reguły, których przesłanki są spełnione
3. **Rozwiązanie konfliktu** – wybór jednej reguły z agendy (strategia)
4. **Wykonanie reguły** – dodanie wniosku do faktów
5. **Nowe fakty + zapis „dlaczego”** – zapamiętujemy, z jakiej reguły wynika wniosek

## Miejsce w projekcie

Karta projektu zakłada modele nienadzorowane (Isolation Forest) i nadzorowane (Random Forest) na zbiorze IEEE-CIS. System regułowy jest warstwą uzupełniającą: daje **wyjaśnialne** uzasadnienie („dlaczego ta transakcja jest podejrzana?”) i pokazuje analitykowi drzewo wywodu. Ostateczną decyzję zawsze podejmuje człowiek (karta projektu, p. 8).

Fakty wejściowe odpowiadają cechom IEEE-CIS. To **założenie modelowe**: kolumny są zanonimizowane, a progi (co to znaczy „wysoka kwota”) trzeba by wyznaczyć na danych. Reguły w notebooku są ilustracyjne i nie zostały wyuczone ze zbioru.

| Fakt wejściowy | Odpowiednik w IEEE-CIS (założenie) |
| --- | --- |
| wysoka_kwota | TransactionAmt powyżej progu |
| kwota_nietypowa_dla_karty | TransactionAmt odbiega od zwykłych kwot danej karty (card1–card6) |
| duza_odleglosc_adresow | dist1–dist2, addr1 |
| inny_kraj | addr2 inny niż zwykle |
| rozne_domeny_email | P_emaildomain różna od R_emaildomain |
| nowe_urzadzenie | DeviceType, DeviceInfo niewidziane wcześniej |
| wysokie_liczniki_c | C1–C14 (zliczenia powiązań) |
| krotki_odstep_od_poprzedniej | D1–D15 (dni od poprzedniego zdarzenia) |
| pierwsza_transakcja_karty | D1–D15 (brak wcześniejszej historii) |
| flagi_m_niezgodne | M1–M9 = F |
| transakcja_nocna | TransactionDT (pora dnia) |
| wiele_malych_transakcji | niska TransactionAmt przy wysokich licznikach C |
| stala_historia_karty | D1–D15 (długa historia karty) |
| tozsamosc_zweryfikowana | id_01–id_38 (wyniki weryfikacji tożsamości) |

## G1. Baza wiedzy

Reguły układają się w łańcuchy: cechy transakcji → anomalia → poziom ryzyka → zalecenie (łańcuchy dwu- i trzypoziomowe). R14 daje drugą drogę do tego samego zalecenia co R7.
R15 jest celowo **konkurencyjna** względem R7 (ta sama przesłanka `wysokie_ryzyko`, inne zalecenie) – służy do demonstracji strategii konfliktu (G2).

In [ ]:
# Baza wiedzy: ocena ryzyka transakcji kartowej. Reguła: IF wszystkie przesłanki THEN wniosek
REGULY = [
    # --- anomalie wyprowadzane z cech transakcji ---
    {'id': 'R1', 'if': {'wysoka_kwota', 'kwota_nietypowa_dla_karty'}, 'then': 'anomalia_kwoty', 'prio': 1},
    {'id': 'R2', 'if': {'duza_odleglosc_adresow', 'inny_kraj'}, 'then': 'anomalia_lokalizacji', 'prio': 1},
    {'id': 'R3', 'if': {'nowe_urzadzenie', 'rozne_domeny_email'}, 'then': 'anomalia_tozsamosci', 'prio': 1},
    {'id': 'R4', 'if': {'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej'}, 'then': 'anomalia_aktywnosci', 'prio': 1},
    # --- poziom ryzyka ---
    {'id': 'R5', 'if': {'anomalia_kwoty', 'anomalia_lokalizacji'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R6', 'if': {'anomalia_aktywnosci', 'anomalia_tozsamosci'}, 'then': 'wysokie_ryzyko', 'prio': 3},
    {'id': 'R8', 'if': {'flagi_m_niezgodne', 'pierwsza_transakcja_karty'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R10', 'if': {'anomalia_lokalizacji', 'transakcja_nocna'}, 'then': 'srednie_ryzyko', 'prio': 2},
    {'id': 'R11', 'if': {'stala_historia_karty', 'tozsamosc_zweryfikowana'}, 'then': 'niskie_ryzyko', 'prio': 1},
    {'id': 'R13', 'if': {'wiele_malych_transakcji', 'pierwsza_transakcja_karty'}, 'then': 'podejrzenie_testowania_karty', 'prio': 2},
    # --- zalecenia dla analityka ---
    {'id': 'R7', 'if': {'wysokie_ryzyko'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    {'id': 'R9', 'if': {'srednie_ryzyko'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 1},
    {'id': 'R12', 'if': {'niskie_ryzyko'}, 'then': 'zalecenie_zatwierdz', 'prio': 1},
    {'id': 'R14', 'if': {'podejrzenie_testowania_karty'}, 'then': 'zalecenie_zablokuj_do_weryfikacji', 'prio': 1},
    # --- reguła KONKURENCYJNA do R7: ta sama przesłanka, inne zalecenie ---
    {'id': 'R15', 'if': {'wysokie_ryzyko', 'tozsamosc_zweryfikowana'}, 'then': 'zalecenie_eskaluj_kontakt_z_klientem', 'prio': 2},
]

# Fakty, które mogą przyjść z danych transakcji (wejście). Służą do wykrywania martwych reguł.
WEJSCIA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
           'nowe_urzadzenie', 'rozne_domeny_email', 'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej',
           'flagi_m_niezgodne', 'pierwsza_transakcja_karty', 'transakcja_nocna', 'stala_historia_karty',
           'tozsamosc_zweryfikowana', 'wiele_malych_transakcji'}


## G1. Wnioskowanie w przód

Każdy cykl: agenda → wybór jednej reguły wg strategii → wykonanie → zapis uzasadnienia. `stop_na_zaleceniu=True` kończy wnioskowanie, gdy pojawi się pierwsze zalecenie.

In [ ]:
def forward_chain(fakty, reguly, strategia='kolejnosc', stop_na_zaleceniu=False, verbose=False):
    '''Wnioskowanie w przód. Zwraca (fakty, powod), gdzie powod[fakt] = (id_reguly, przeslanki).
    strategia: 'kolejnosc' (pierwsza reguła z agendy) lub 'priorytet' (najwyższy prio; remis -> kolejność).'''
    fakty = set(fakty)
    powod = {}
    krok = 0
    while True:
        # 1) dopasowanie: agenda = reguły, których przesłanki są w faktach, a wniosku jeszcze nie ma
        agenda = [r for r in reguly if r['if'] <= fakty and r['then'] not in fakty]
        if not agenda:
            break
        # 2) rozwiązanie konfliktu
        if strategia == 'kolejnosc':
            r = agenda[0]
        elif strategia == 'priorytet':
            r = max(agenda, key=lambda x: x['prio'])   # max zwraca pierwszą z najwyższych
        else:
            raise ValueError('nieznana strategia: ' + strategia)
        # 3) wykonanie + zapis wniosku i uzasadnienia
        krok += 1
        fakty.add(r['then'])
        powod[r['then']] = (r['id'], sorted(r['if']))
        if verbose:
            print('krok', krok, r['id'], sorted(r['if']), '->', r['then'], '| agenda:', [a['id'] for a in agenda])
        if stop_na_zaleceniu and r['then'].startswith('zalecenie_'):
            break
    return fakty, powod


def wyjasnij(fakt, powod, poziom=0):
    '''Odpowiada na pytanie dlaczego: drzewo wywodu wstecz po zapisanych uzasadnieniach.'''
    wciecie = '  ' * poziom
    if fakt not in powod:
        print(wciecie + '- ' + fakt + '  [fakt startowy]')
        return
    rid, przeslanki = powod[fakt]
    print(wciecie + '- ' + fakt + '  [z ' + rid + ': ' + str(przeslanki) + ']')
    for p in przeslanki:
        wyjasnij(p, powod, poziom + 1)


# Przykład: transakcja o nietypowej kwocie i lokalizacji
wynik, _ = forward_chain({'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'},
                         REGULY, verbose=True)
print(sorted(wynik))


## G2. Konflikty i wyjaśnianie „dlaczego?”

Ta sama transakcja z zweryfikowaną tożsamością klienta, dwie strategie. Przy `stop_na_zaleceniu=True` wynik zależy wyłącznie od strategii rozwiązywania konfliktu: `kolejnosc` daje blokadę (R7), `priorytet` eskalację z kontaktem z klientem (R15).

In [ ]:
TRANSAKCJA = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj',
              'tozsamosc_zweryfikowana'}

for strat in ['kolejnosc', 'priorytet']:
    print('=== strategia:', strat, '(stop na zaleceniu) ===')
    fakty, powod = forward_chain(TRANSAKCJA, REGULY, strategia=strat, stop_na_zaleceniu=True, verbose=True)
    zalecenia = sorted(f for f in fakty if f.startswith('zalecenie_'))
    print('zalecenie:', zalecenia)
    print('--- dlaczego? ---')
    for z in zalecenia:
        wyjasnij(z, powod)
    print()


## G2/G3. Analiza bazy: reguły martwe i konfliktowe

- **Martwa** – reguła, której przesłanka nie jest wnioskiem żadnej reguły ani faktem wejściowym (nigdy nie zostanie odpalona).
- **Konfliktowa** – dwie reguły, których przesłanki się zawierają (mogą być aktywne jednocześnie), a wnioski są różne.

Uwaga: para R7–R15 jest celowa (demo strategii), a nie błąd bazy.

In [ ]:
def analiza_bazy(reguly, wejscia=WEJSCIA):
    wyprodukowane = set(r['then'] for r in reguly)
    martwe = [r['id'] for r in reguly if any(p not in wyprodukowane and p not in wejscia for p in r['if'])]
    konflikty = []
    for i, a in enumerate(reguly):
        for b in reguly[i + 1:]:
            if a['then'] != b['then'] and (a['if'] <= b['if'] or b['if'] <= a['if']):
                konflikty.append((a['id'], b['id']))
    return martwe, konflikty

martwe, konflikty = analiza_bazy(REGULY)
print('martwe reguły:', martwe)
print('konfliktowe pary:', konflikty)

# Test wykrywania: dopisana reguła z przesłanką, której nic nie produkuje
test = REGULY + [{'id': 'R99', 'if': {'cecha_spoza_zbioru'}, 'then': 'zalecenie_zatwierdz', 'prio': 1}]
print('z R99 -> martwe:', analiza_bazy(test)[0])


## G3. Testy bazy wiedzy

Scenariusze: `musi` – fakty, które muszą wyjść; `nie_moze` – fakty, które nie mogą wyjść. Scenariusz S6 pokazuje, że sama wysoka kwota (jak w obecnej regule progowej z karty projektu) nie wystarcza do blokady.

In [ ]:
SCENARIUSZE = [
    {'nazwa': 'S1 nietypowa kwota + lokalizacja -> blokada', 'fakty': {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'},
     'musi': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'},
     'nie_moze': {'zalecenie_eskaluj_kontakt_z_klientem', 'zalecenie_zatwierdz'}},
    {'nazwa': 'S2 aktywność + tożsamość -> blokada', 'fakty': {'wysokie_liczniki_c', 'krotki_odstep_od_poprzedniej', 'nowe_urzadzenie', 'rozne_domeny_email'},
     'musi': {'anomalia_aktywnosci', 'anomalia_tozsamosci', 'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': set()},
    {'nazwa': 'S3 niezgodne flagi M + nowa karta -> eskalacja', 'fakty': {'flagi_m_niezgodne', 'pierwsza_transakcja_karty'},
     'musi': {'srednie_ryzyko', 'zalecenie_eskaluj_kontakt_z_klientem'},
     'nie_moze': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S4 stała historia + zweryfikowana tożsamość -> zatwierdź', 'fakty': {'stala_historia_karty', 'tozsamosc_zweryfikowana'},
     'musi': {'niskie_ryzyko', 'zalecenie_zatwierdz'},
     'nie_moze': {'wysokie_ryzyko', 'srednie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S5 testowanie karty -> blokada', 'fakty': {'wiele_malych_transakcji', 'pierwsza_transakcja_karty'},
     'musi': {'podejrzenie_testowania_karty', 'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': {'zalecenie_zatwierdz'}},
    {'nazwa': 'S6 sama wysoka kwota -> brak blokady', 'fakty': {'wysoka_kwota'},
     'musi': set(), 'nie_moze': {'anomalia_kwoty', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S7 pusta lista faktów -> brak zaleceń', 'fakty': set(),
     'musi': set(), 'nie_moze': {'zalecenie_zablokuj_do_weryfikacji', 'zalecenie_eskaluj_kontakt_z_klientem', 'zalecenie_zatwierdz'}},
    {'nazwa': 'S8 anomalia lokalizacji w nocy -> eskalacja', 'fakty': {'duza_odleglosc_adresow', 'inny_kraj', 'transakcja_nocna'},
     'musi': {'anomalia_lokalizacji', 'srednie_ryzyko', 'zalecenie_eskaluj_kontakt_z_klientem'}, 'nie_moze': {'wysokie_ryzyko'}},
    {'nazwa': 'S9 sama anomalia tożsamości -> brak wysokiego ryzyka', 'fakty': {'nowe_urzadzenie', 'rozne_domeny_email'},
     'musi': {'anomalia_tozsamosci'}, 'nie_moze': {'wysokie_ryzyko', 'zalecenie_zablokuj_do_weryfikacji'}},
    {'nazwa': 'S10 sama nowa karta -> brak testowania karty', 'fakty': {'pierwsza_transakcja_karty'},
     'musi': set(), 'nie_moze': {'podejrzenie_testowania_karty', 'srednie_ryzyko'}},
    {'nazwa': 'S11 konflikt, kolejność -> blokada', 'fakty': TRANSAKCJA, 'strategia': 'kolejnosc', 'stop': True,
     'musi': {'zalecenie_zablokuj_do_weryfikacji'}, 'nie_moze': {'zalecenie_eskaluj_kontakt_z_klientem'}},
    {'nazwa': 'S12 konflikt, priorytet -> eskalacja', 'fakty': TRANSAKCJA, 'strategia': 'priorytet', 'stop': True,
     'musi': {'zalecenie_eskaluj_kontakt_z_klientem'}, 'nie_moze': {'zalecenie_zablokuj_do_weryfikacji'}},
]

def uruchom_testy(scenariusze, reguly=REGULY):
    zaliczone = 0
    for s in scenariusze:
        fakty, _ = forward_chain(s['fakty'], reguly, strategia=s.get('strategia', 'kolejnosc'),
                                 stop_na_zaleceniu=s.get('stop', False))
        ok = s['musi'] <= fakty and not (s['nie_moze'] & fakty)
        zaliczone += ok
        print('OK   ' if ok else 'BŁĄD ', s['nazwa'])
    print('Wynik:', zaliczone, '/', len(scenariusze))
    return zaliczone == len(scenariusze)

assert uruchom_testy(SCENARIUSZE), 'Są niezaliczone testy'


## B1. Wnioskowanie wstecz (od celu do faktów)

Wnioskowanie w przód pyta: „co wynika z cech tej transakcji?”. Wnioskowanie wstecz pyta: „czy zalecenie (cel) jest uzasadnione i jakich cech brakuje, żeby je udowodnić?”.

- **Kiedy w przód:** mamy komplet cech transakcji i chcemy wszystkie wnioski (ocena każdej transakcji w kolejce).
- **Kiedy wstecz:** analityk ma hipotezę (np. „czy to testowanie karty?”) i chce wiedzieć, jakie dane dopytać lub sprawdzić. Wynik to lista brakujących faktów, a nie pełny zbiór wniosków.
- Wnioskowanie wstecz odrzuca cykle (cel pojawia się wśród własnych przesłanek), żeby nie zapętlić się.

In [ ]:
def backward_chain(cel, reguly, fakty, glebokosc=0, odwiedzone=frozenset()):
    '''Wnioskowanie wstecz od celu. Zwraca (udowodniony, slad). Cykle są odrzucane.'''
    fakty = set(fakty)
    if cel in fakty:
        return True, [(glebokosc, cel, 'fakt')]
    if cel in odwiedzone:
        return False, [(glebokosc, cel, 'cykl - odrzucone')]
    odwiedzone = odwiedzone | {cel}
    slad_porazek = []
    for r in reguly:                       # alternatywy: reguły z tym wnioskiem
        if r['then'] != cel:
            continue
        podslad = [(glebokosc, cel, 'z ' + r['id'] + ' ' + str(sorted(r['if'])))]
        ok = True
        for p in sorted(r['if']):          # podcele = przesłanki
            udowodniony, s = backward_chain(p, reguly, fakty, glebokosc + 1, odwiedzone)
            podslad += s
            if not udowodniony:
                ok = False
                break
        if ok:
            return True, podslad
        slad_porazek += podslad + [(glebokosc, cel, r['id'] + ' odrzucona')]
    if not slad_porazek:
        slad_porazek = [(glebokosc, cel, 'brak reguły i brak faktu')]
    return False, slad_porazek


def pokaz_slad(slad):
    for glebokosc, cel, opis in slad:
        print('  ' * glebokosc + '- ' + cel + '  [' + opis + ']')


def brakujace_fakty(cel, reguly, fakty, odwiedzone=frozenset()):
    '''Najmniejszy zbiór brakujących faktów wejściowych, potrzebny do udowodnienia celu.'''
    if cel in fakty or cel in odwiedzone:
        return set()
    regs = [r for r in reguly if r['then'] == cel]
    if not regs:
        return {cel}
    kandydaci = []
    for r in regs:
        braki = set()
        for p in r['if']:
            braki |= brakujace_fakty(p, reguly, fakty, odwiedzone | {cel})
        kandydaci.append(braki)
    return min(kandydaci, key=len)


In [ ]:
# Transakcja z nietypową kwotą i dużą odległością adresów, ale bez informacji o kraju
FAKTY_B1 = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow'}

for cel in ['anomalia_kwoty', 'zalecenie_zablokuj_do_weryfikacji', 'zalecenie_zatwierdz']:
    ok, slad = backward_chain(cel, REGULY, FAKTY_B1)
    print('CEL:', cel, '| udowodniony:', ok)
    pokaz_slad(slad)
    print('brakujące fakty:', sorted(brakujace_fakty(cel, REGULY, FAKTY_B1)))
    print()


In [ ]:
# Zgodność: wstecz i w przód dają ten sam wynik dla każdego faktu i każdego scenariusza testowego
cele = sorted({r['then'] for r in REGULY} | WEJSCIA)
for s in SCENARIUSZE:
    pelne, _ = forward_chain(s['fakty'], REGULY)   # pełne domknięcie w przód
    for cel in cele:
        ok_wstecz, _ = backward_chain(cel, REGULY, s['fakty'])
        assert ok_wstecz == (cel in pelne), (s['nazwa'], cel)
print('Zgodność wstecz/w przód: OK dla', len(SCENARIUSZE), 'scenariuszy i', len(cele), 'faktów')


## P1. Reguły jako guardrails dla LLM (trend 2026, bonus)

Wyobraźmy sobie asystenta LLM, który podpowiada analitykowi akcję dla transakcji. Zanim akcja zostanie przyjęta, deterministyczne reguły sprawdzają:

1. **Wejście:** czy fakty pochodzą z bazy wiedzy (odrzucamy fakty wstrzyknięte, np. „klient_vip_zatwierdzony”).
2. **Akcje zakazane:** np. trwałe zamknięcie konta, usunięcie historii transakcji.
3. **Zgodność z bazą wiedzy:** akcja musi wynikać z cech transakcji przez wnioskowanie w przód. Jeśli nie wynika, to halucynacja i akcja jest odrzucana.
4. **Nadzór człowieka:** każda akcja wymaga zatwierdzenia analityka (zgodnie z p. 8 karty projektu: model nie podejmuje autonomicznych decyzji).

Zasada: LLM proponuje, reguły decydują, analityk zatwierdza. Tu propozycja modelu jest symulowana słownikiem, więc notebook działa offline. W praktyce propozycję zwróciłby model przez API lub przez MCP.

In [ ]:
# P1: reguły jako guardrails dla propozycji LLM (LLM proponuje, reguły decydują, analityk zatwierdza)
ZAKAZANE_AKCJE = {'trwale_zamknij_konto', 'usun_historie_transakcji', 'zablokuj_bez_weryfikacji_analityka'}
AKCJA_NA_ZALECENIE = {
    'zablokuj_do_weryfikacji': 'zalecenie_zablokuj_do_weryfikacji',
    'eskaluj_kontakt_z_klientem': 'zalecenie_eskaluj_kontakt_z_klientem',
    'zatwierdz': 'zalecenie_zatwierdz',
}
WYMAGANE_POLA = {akcja: ['zatwierdzenie_analityka'] for akcja in AKCJA_NA_ZALECENIE}


def guardrail(propozycja, fakty_transakcji, reguly=REGULY):
    '''Deterministyczna kontrola propozycji LLM. Zwraca (dozwolona, lista_powodow).'''
    powody = []
    # 1) wejście: fakty spoza bazy wiedzy są odrzucane
    obce = set(fakty_transakcji) - WEJSCIA
    if obce:
        powody.append('nieznane fakty wejściowe: ' + str(sorted(obce)))
    akcja = propozycja.get('akcja')
    # 2) akcje zakazane
    if akcja in ZAKAZANE_AKCJE:
        powody.append('akcja zakazana: ' + akcja)
    # 3) zgodność z bazą wiedzy: zalecenie musi wynikać z cech transakcji
    elif akcja not in AKCJA_NA_ZALECENIE:
        powody.append('akcja spoza bazy wiedzy: ' + str(akcja))
    else:
        wyprowadzone, _ = forward_chain(fakty_transakcji, reguly)
        wymagane = AKCJA_NA_ZALECENIE[akcja]
        if wymagane not in wyprowadzone:
            powody.append('brak wsparcia w bazie: ' + wymagane + ' nie wynika z faktów')
    # 4) nadzór człowieka: wymagane pola
    for pole in WYMAGANE_POLA.get(akcja, []):
        if not propozycja.get(pole):
            powody.append('brak wymaganego pola: ' + pole)
    return len(powody) == 0, powody


In [ ]:
FAKTY_WYSOKIE = {'wysoka_kwota', 'kwota_nietypowa_dla_karty', 'duza_odleglosc_adresow', 'inny_kraj'}
FAKTY_ZAUFANE = {'stala_historia_karty', 'tozsamosc_zweryfikowana'}

PRZYPADKI = [
    ('poprawna blokada', {'akcja': 'zablokuj_do_weryfikacji', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, True),
    ('poprawne zatwierdzenie', {'akcja': 'zatwierdz', 'zatwierdzenie_analityka': True}, FAKTY_ZAUFANE, True),
    ('halucynacja: zatwierdź ryzykowną', {'akcja': 'zatwierdz', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
    ('brak zatwierdzenia analityka', {'akcja': 'zablokuj_do_weryfikacji'}, FAKTY_WYSOKIE, False),
    ('akcja zakazana', {'akcja': 'trwale_zamknij_konto', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
    ('wstrzyknięty fakt', {'akcja': 'zablokuj_do_weryfikacji', 'zatwierdzenie_analityka': True},
     FAKTY_WYSOKIE | {'klient_vip_zatwierdzony'}, False),
    ('akcja spoza bazy', {'akcja': 'zwroc_srodki', 'zatwierdzenie_analityka': True}, FAKTY_WYSOKIE, False),
]

zaliczone = 0
for nazwa, prop, fakty, oczekiwane in PRZYPADKI:
    dozwolona, powody = guardrail(prop, fakty)
    zgodne = dozwolona == oczekiwane
    zaliczone += zgodne
    print('OK   ' if zgodne else 'BŁĄD ', nazwa, '->', 'dozwolona' if dozwolona else 'odrzucona', powody)
print('Wynik:', zaliczone, '/', len(PRZYPADKI))
assert zaliczone == len(PRZYPADKI)


## Wnioski

- Strategia rozwiązywania konfliktu zmienia wynik, gdy na agendzie są reguły o tej samej przesłance i różnych wnioskach (R7 vs R15). Dla transakcji z zweryfikowaną tożsamością `kolejnosc` daje blokadę do weryfikacji, a `priorytet` eskalację z kontaktem z klientem.
- Analiza bazy: brak martwych reguł (`martwe: []`). Wykryto jedną parę konfliktową R7–R15, która jest celowa i służy demonstracji strategii. Test z dopisaną regułą R99 (przesłanka `cecha_spoza_zbioru`, której nic nie produkuje) poprawnie zgłasza regułę jako martwą.
- Wszystkie 12 scenariuszy testowych przechodzi (12/12), w tym wnioski wielopoziomowe i przypadki negatywne. Scenariusz S6 pokazuje, że sama wysoka kwota nie prowadzi do blokady, w przeciwieństwie do prostej reguły progowej z karty projektu.
- Rola w projekcie: reguły dają wyjaśnialną warstwę obok modeli ML (Isolation Forest, Random Forest). Drzewo wywodu odpowiada analitykowi na pytanie, dlaczego transakcja jest podejrzana, a decyzja zostaje po stronie człowieka.
- Ograniczenia: reguły są ilustracyjne i nie wyuczone z IEEE-CIS, a progi cech trzeba wyznaczyć na danych. Cech V1–V339 nie da się użyć w regułach, bo ich znaczenie jest niejawne. W każdym cyklu odpalana jest jedna reguła, wykrywanie konfliktów jest przybliżone, a baza nie obsługuje negacji ani wag pewności.
- Dalsze kroki: wyprowadzać fakty z wyników modeli (np. wynik ryzyka powyżej progu) i porównać jakość kolejki z regułami i bez nich na PR AUC oraz precision@k.


## Użycie AI

Notebook (zadania G1–G3, B1 i P1) oraz dopasowanie go do tematu projektu (wykrywanie oszustw w transakcjach kartowych) wygenerowano z pomocą Claude Code (modele Haiku 5.5 i Sonnet 5.5): kod bazy wiedzy, wnioskowania w przód i wstecz, analizy bazy, scenariuszy testowych, guardrails dla LLM oraz szkic wniosków. Temat i założenia wzięto z pliku z kartą projektu.

Co sprawdzono: wszystkie komórki kodu uruchomiono od początku do końca bez błędów. Przeszło 12/12 scenariuszy testowych, 7/7 przypadków guardrails oraz kontrola zgodności wnioskowania wstecz i w przód. Ślady wnioskowania (kolejność reguł, wybór w konflikcie, drzewa „dlaczego?”) sprawdzono na wyniku uruchomienia.

Ograniczenia: reguły i mapowanie faktów na kolumny IEEE-CIS to założenia ilustracyjne, niezweryfikowane na danych. W P1 propozycja modelu jest symulowana słownikiem, bez prawdziwego wywołania LLM przez API.

Dobór reguł, treść wniosków i odpowiedzi na pytania kontrolne należą do autorów sprawozdania.
